
# TinyCenterSpeed (Object + Free) — W&B Sweep
사용자 코드의 **두 경로(train/val 각각 obj + free)**를 그대로 유지하면서, W&B **Sweep** 기능을 추가했습니다.

### 포함 기능
- 기존 파이프라인 보존(ZeroTargetWrapper, 4개 데이터 경로 병합)
- **Bayesian sweep**로 하이퍼파라미터 탐색
- `lr/alpha` 변화에 따른 **loss 곡선 시각화** (W&B UI)
- **Hyperparameter importance / correlation** (W&B 대시보드)
- 여러 run 비교, **Best run 재학습**(선택)


In [1]:

#!/usr/bin/env python3
import os, sys, math, random, datetime, json, gc, time
import numpy as np
import torch
from torch.utils.data import DataLoader, ConcatDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 프로젝트 경로(필요 시 수정)
current_dir = os.path.dirname(os.path.abspath(''))
two_up_dir = os.path.dirname(os.path.dirname(current_dir))
if two_up_dir not in sys.path:
    sys.path.append(two_up_dir)

# 모델/데이터셋/로스 임포트
from TinyCenterSpeed.src.models.CenterSpeed import CenterSpeedDense
from TinyCenterSpeed.dataset.CenterSpeed_dataset import CenterSpeedDataset, RandomRotation, RandomFlip
from TinyCenterSpeed.src.models.losses import *  # 필요 시 내부 함수 사용

# W&B 설정
use_wandb = True
project_name = "TinyCenterSpeed_redbull_free_obj"
run_name = "train_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    import wandb
    if use_wandb:
        if os.environ.get("WANDB_MODE","").lower() == "offline":
            wandb.init(project=project_name, name=run_name, mode="offline")
            wandb.finish()
        else:
            try:
                wandb.login(anonymous="allow")
            except Exception as e:
                print("[wandb] login 실패, offline으로 진행:", e)
                os.environ["WANDB_MODE"] = "offline"
except Exception as e:
    print(f"[wandb] 사용 불가: {type(e).__name__}: {e}")
    use_wandb = False
    wandb = None

# 재현성/디바이스
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.backends.cudnn.benchmark = True


wandb: Currently logged in as: whdaudpark (whdaudpark-dongguk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


In [2]:

# ===== 기본 경로 (사용자 코드 유지) =====
train_obj_path  = "/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT"
train_free_path = "/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT_free"
val_obj_path    = "/home/harry/sim_ws/src/f1tenth_gym_ros/Val_set"
val_free_path   = "/home/harry/sim_ws/src/f1tenth_gym_ros/Val_set_free"

# ===== 기본 하이퍼파라미터 (sweep에서 바뀔 수 있음) =====
DEFAULTS = {
    "image_size": 128,
    "pixelsize": 0.1,
    "sigma_px": 1.0,
    "epochs": 100,
    "batch_size": 32,
    "learning_rate": 5e-4,
    "alpha": 0.95,
    "scheduler_factor": 0.5,
    "scheduler_patience": 8,
    "scheduler_threshold": 0.005,
    "augment_rotation": 45,     # 0 또는 45
    "augment_flip": 0.5,       # 0.0 또는 0.5
    "weight_decay": 1e-5,       # 0 ~ 1e-3  역전파 과정에서 기울기 폭파를 막아서(기울기 비율 줄임) 파라미터 업데이트 폭발 안하게
    "optimizer": "Adam",       # Adam / AdamW
    "clip_grad_norm": 1.0,     # 0 / 1 / 5  기울기 폭방 방지를 위한 기울기의 크기 제한
    "num_workers_cap": 8,      # 최대 워커 수
}

SAVE_DIR = "/home/harry/ros2_ws/src/TinyCenterSpeed/src/pt"
os.makedirs(SAVE_DIR, exist_ok=True)

# DataLoader 최적화
def compute_loader_params(num_workers_cap=8):
    NUM_WORKERS = min(os.cpu_count() or 4, int(num_workers_cap))
    PIN_MEMORY  = torch.cuda.is_available()
    PERSIST     = NUM_WORKERS > 0
    return NUM_WORKERS, PIN_MEMORY, PERSIST


In [3]:

# 객체 없음 데이터에서 GT를 0으로 강제하는 래퍼
class ZeroTargetWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        inputs, gts, data_vec, dense_feats, is_free = self.base[idx]
        gts = torch.zeros_like(gts)
        dense_feats = torch.zeros_like(dense_feats)
        return inputs, gts, data_vec, dense_feats, is_free

from torchvision import transforms as T

def make_dataset(root_dir, image_size, pixelsize, sigma_px, augment_rotation=0, augment_flip=0.0):
    tfs = []
    if augment_rotation:
        tfs.append(RandomRotation(int(augment_rotation), image_size=int(image_size)))
    if augment_flip and float(augment_flip) > 0:
        tfs.append(RandomFlip(float(augment_flip)))
    transform = T.Compose(tfs) if len(tfs) > 0 else None

    ds = CenterSpeedDataset(dataset_path=root_dir, transform=transform, dense=True)
    # 픽셀/이미지/σ 설정
    try:
        ds.change_image_size(int(image_size))
    except Exception:
        ds.image_size = int(image_size)
    try:
        ds.change_pixel_size(float(pixelsize))
    except Exception:
        ds.pixelsize = float(pixelsize)
    ds.sx = float(sigma_px)
    ds.sy = float(sigma_px)
    return ds

def build_loaders(cfg):
    # Train datasets
    train_obj_dataset  = make_dataset(
        train_obj_path, cfg["image_size"], cfg["pixelsize"], cfg["sigma_px"],
        cfg["augment_rotation"], cfg["augment_flip"]
    )
    train_free_dataset = make_dataset(
        train_free_path, cfg["image_size"], cfg["pixelsize"], cfg["sigma_px"],
        cfg["augment_rotation"], cfg["augment_flip"]
    )
    train_free_dataset = ZeroTargetWrapper(train_free_dataset)
    train_dataset = ConcatDataset([train_obj_dataset, train_free_dataset])

    # Val datasets
    val_obj_dataset  = make_dataset(
        val_obj_path, cfg["image_size"], cfg["pixelsize"], cfg["sigma_px"],
        0, 0.0  # 검증은 보통 augmentation 비활성화
    )
    val_free_dataset = make_dataset(
        val_free_path, cfg["image_size"], cfg["pixelsize"], cfg["sigma_px"],
        0, 0.0
    )
    val_free_dataset = ZeroTargetWrapper(val_free_dataset)
    val_dataset = ConcatDataset([val_obj_dataset, val_free_dataset])

    # DataLoader
    NUM_WORKERS, PIN_MEMORY, PERSIST = compute_loader_params(cfg.get("num_workers_cap", 8))

    def worker_init_fn(_):
        try:
            import torch, os
            torch.set_num_threads(1)
            os.environ.setdefault("OMP_NUM_THREADS", "1")
            os.environ.setdefault("MKL_NUM_THREADS", "1")
        except Exception:
            pass

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=int(cfg["batch_size"]),
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSIST,
        prefetch_factor=4,
        drop_last=True,
        worker_init_fn=worker_init_fn,
    )

    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=int(cfg["batch_size"]),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSIST,
        prefetch_factor=2,
        drop_last=False,
        worker_init_fn=worker_init_fn,
    )

    print("train_obj:", len(train_obj_dataset), 
          "train_free:", len(train_free_dataset), 
          "=> train_total:", len(train_dataset))
    print("val_obj:", len(val_obj_dataset), 
          "val_free:", len(val_free_dataset), 
          "=> val_total:", len(val_dataset))

    return train_loader, val_loader


In [4]:

# 출력 [B,4,H,W], gts [B,H,W], dense [B,H,W,3]
def dense_loss(output, gt_heatmap, gt_dense_data, is_free, alpha=0.95):
    preds = output.permute(0,2,3,1)  # [B,H,W,4]
    w = gt_heatmap.unsqueeze(-1)     # [B,H,W,1]
    loss_occ   = (alpha     * (1 + w) * (preds[...,0:1] - gt_heatmap.unsqueeze(-1))**2).sum()
    loss_dense = ((1-alpha) * (1 + w) * (preds[...,1:]  - gt_dense_data)**2).sum()
    batch_size = output.shape[0]
    return (loss_occ + loss_dense) / batch_size


In [5]:
def train_one_run(config: dict | None = None):
    # 1) 먼저 W&B run 시작 (DEFAULTS를 기본 설정으로 올려두면 대시보드에서 전부 보임)
    run = None
    if 'wandb' in globals() and wandb is not None:
        run = wandb.init(project=project_name, config=DEFAULTS, reinit=True)

    # 2) cfg 병합 순서: DEFAULTS -> (wandb.sweep 샘플) -> (외부 전달 config)
    cfg = DEFAULTS.copy()

    # sweep/agent 실행 시: init 이후에만 wandb.config 접근 가능
    if run is not None:
        cfg.update(dict(wandb.config))   # ← 여기서 샘플링된 하이퍼파라미터가 들어옴

    # 수동 호출 시 넘긴 config가 있으면 최종 오버라이드
    if config is not None:
        cfg.update(dict(config))         # dict()로 캐스팅해 안전하게 반영

    # ===== 아래부터는 네 원래 코드와 동일 =====
    train_loader, val_loader = build_loaders(cfg)
    model = CenterSpeedDense(input_channels=4, image_size=int(cfg["image_size"])).to(device)

    # Optimizer
    if cfg["optimizer"] == "AdamW":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=float(cfg["learning_rate"]),
            weight_decay=float(cfg["weight_decay"])
        )
    else:
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=float(cfg["learning_rate"]),
            weight_decay=float(cfg["weight_decay"])
        )

    # Scheduler
    scheduler = ReduceLROnPlateau(
        optimizer, mode="min",
        factor=float(cfg["scheduler_factor"]),
        patience=int(cfg["scheduler_patience"]),
        threshold=float(cfg["scheduler_threshold"])
    )

    best_val = float('inf')
    best_path = None

    for epoch in range(1, int(cfg["epochs"]) + 1):
        model.train()
        running = 0.0
        for batch in train_loader:
            inputs, gts, data_vec, dense_feats, is_free = batch
            inputs      = inputs.to(device, non_blocking=True)
            gts         = gts.to(device, non_blocking=True)
            dense_feats = dense_feats.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            out = model(inputs)
            loss = dense_loss(out, gts, dense_feats, is_free, alpha=float(cfg["alpha"]))
            loss.backward()

            if float(cfg["clip_grad_norm"]) > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg["clip_grad_norm"]))

            # (선택) 폭주/NaN 가드
            if not torch.isfinite(loss):
                print("[WARN] Non-finite loss detected, aborting this run.")
                if run is not None:
                    wandb.summary["aborted_reason"] = "non-finite loss"
                return float('inf'), None, cfg

            optimizer.step()
            running += loss.item()

        train_loss = running / max(1, len(train_loader))

        # Validation
        model.eval()
        v_running = 0.0
        with torch.no_grad():
            for batch in val_loader:
                inputs, gts, data_vec, dense_feats, is_free = batch
                inputs      = inputs.to(device, non_blocking=True)
                gts         = gts.to(device, non_blocking=True)
                dense_feats = dense_feats.to(device, non_blocking=True)
                out = model(inputs)
                v_loss = dense_loss(out, gts, dense_feats, is_free, alpha=float(cfg["alpha"]))
                v_running += v_loss.item()

        val_loss = v_running / max(1, len(val_loader)) if len(val_loader) > 0 else train_loss

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"[{epoch:03d}/{cfg['epochs']}] train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | lr={current_lr:.3e}")
        if run is not None:
            wandb.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "lr": current_lr
            })

        if val_loss < best_val:
            best_val = val_loss
            best_path = os.path.join(SAVE_DIR, f"redbull_objfree_sweep_{wandb.run.id if run else 'offline'}_e{epoch}.pt")
            torch.save(model.state_dict(), best_path)
            print(f"  ↳ Best model saved: {best_path} (val={best_val:.6f})")
            if run is not None:
                art = wandb.Artifact("best_model", type="model")
                art.add_file(best_path)
                wandb.log_artifact(art)

    if run is not None:
        wandb.summary["best_val_loss"] = best_val
        if best_path:
            wandb.summary["best_model_path"] = best_path
        wandb.finish()

    last_path = os.path.join(SAVE_DIR, f"redbull_objfree_last_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.pt")
    torch.save(model.state_dict(), last_path)
    print(f"Training finished. Model saved at {last_path}")

    return best_val, best_path, cfg


## 0) 단일 실행 (환경 점검용)

In [6]:

# _ = train_one_run(DEFAULTS)  # 필요 시 주석 해제
print("단일 실행은 필요 시 주석 해제 후 사용하세요.")


단일 실행은 필요 시 주석 해제 후 사용하세요.


## 1) Sweep 설정 (Bayesian Search)

In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "epochs": {"values": [30, 50, 80, 100]},
        "batch_size": {"values": [32, 64]},

        # ✅ 값 구간 그대로 로그-균등 샘플
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 5e-4
        },
        # "weight_decay": {
        #     "distribution": "log_uniform_values",
        #     "min": 1e-8,
        #     "max": 1e-3
        # },

        "weight_decay": {"values": [0, 1e-8, 1e-4]},
        "alpha": {"min": 0.90, "max": 0.99},
        "optimizer": {"values": ["Adam", "AdamW"]},
        "clip_grad_norm": {"values": [0.0]},
        "scheduler_factor": {"min": 0.3, "max": 0.8},
        "scheduler_patience": {"values": [5, 6, 8, 10]},
        "scheduler_threshold": {"values": [0.001, 0.003, 0.005, 0.01]},
        "image_size": {"values": [128]},
        "pixelsize": {"values": [0.1]},
        "sigma_px": {"min": 1.0, "max": 3.0},
        "augment_rotation": {"values": [45]},
        "augment_flip": {"values": [0.5]},
        "num_workers_cap": {"values": [4, 8]},
    },
    "early_terminate": {"type": "hyperband", "s": 2, "max_iter": 20, "eta": 3}
}


## 2) Sweep 실행 (에이전트)

In [8]:
SWEEP_RUNS = 10

if 'wandb' in globals() and wandb is not None and os.environ.get("WANDB_MODE","").lower() != "offline":
    sweep_id = wandb.sweep(sweep=sweep_config, project=project_name)
    print("Sweep ID:", sweep_id)

    def _agent():
        # ❌ train_one_run(wandb.config)  (금지)
        # ✅ init은 train_one_run 내부에서 처리
        train_one_run()

    wandb.agent(sweep_id, function=_agent, count=SWEEP_RUNS)
else:
    print("W&B sweep은 온라인 모드에서만 동작합니다. (지금은 offline / 미설치)")


Create sweep with ID: vz1wz8z9
Sweep URL: https://wandb.ai/whdaudpark-dongguk-university/TinyCenterSpeed_redbull_free_obj/sweeps/vz1wz8z9
Sweep ID: vz1wz8z9


wandb: Agent Starting Run: k1r3jxmt with config:
wandb: 	alpha: 0.9045332778227424
wandb: 	augment_flip: 0.5
wandb: 	augment_rotation: 0
wandb: 	batch_size: 64
wandb: 	clip_grad_norm: 0
wandb: 	epochs: 100
wandb: 	image_size: 128
wandb: 	learning_rate: 0.0001909954462779474
wandb: 	num_workers_cap: 4
wandb: 	optimizer: AdamW
wandb: 	pixelsize: 0.1
wandb: 	scheduler_factor: 0.4752995368272592
wandb: 	scheduler_patience: 10
wandb: 	scheduler_threshold: 0.005
wandb: 	sigma_px: 1.653885092078947
wandb: 	weight_decay: 0.0004542990068415923


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs1.csv
Entries     :  14576
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs2.csv
Entries     :  11293
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs3.csv
Entries     :  8009
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs4.csv
Entries     :  3813
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs5.csv
Entries     :  4428
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs1.csv
Entries     :  2361
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs10.csv
Entries     :  842
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs2.csv
Entries     :  2143
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs3.csv
Entries     :  1945
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_ob

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇█
lr,████████████████████▄▄▄▄▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▃▄█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_model_path,/home/harry/ros2_ws/...
best_val_loss,6.85921
epoch,100
lr,0.0
train_loss,3.7554
val_loss,7.27826


...

Training finished. Model saved at /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/redbull_objfree_last_20250816_113828.pt


wandb: Agent Starting Run: lntth5i1 with config:
wandb: 	alpha: 0.9051454833017968
wandb: 	augment_flip: 0.5
wandb: 	augment_rotation: 0
wandb: 	batch_size: 32
wandb: 	clip_grad_norm: 1
wandb: 	epochs: 100
wandb: 	image_size: 128
wandb: 	learning_rate: 0.00030848526893760385
wandb: 	num_workers_cap: 8
wandb: 	optimizer: AdamW
wandb: 	pixelsize: 0.1
wandb: 	scheduler_factor: 0.3030488021141989
wandb: 	scheduler_patience: 10
wandb: 	scheduler_threshold: 0.005
wandb: 	sigma_px: 1.599670239404393
wandb: 	weight_decay: 1.09140618635608e-05


Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs1.csv
Entries     :  14576
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs2.csv
Entries     :  11293
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs3.csv
Entries     :  8009
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs4.csv
Entries     :  3813
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs5.csv
Entries     :  4428
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs1.csv
Entries     :  2361
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs10.csv
Entries     :  842
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs2.csv
Entries     :  2143
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs3.csv
Entries     :  1945
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_ob

Traceback (most recent call last):
  File "/home/harry/.local/lib/python3.10/site-packages/wandb/agents/pyagent.py", line 297, in _run_job
    self._function()
  File "/tmp/ipykernel_983556/2773494514.py", line 10, in _agent
    train_one_run()
  File "/tmp/ipykernel_983556/1516587179.py", line 65, in train_one_run
    if not torch.isfinite(loss):
Exception



epoch,▁█
lr,▁▁
train_loss,█▁
val_loss,█▁
epoch,2
lr,0.00031
train_loss,803.22802
val_loss,152.66182


## 3) Best Run 재실행 (선택)

In [ ]:

# 온라인 모드에서 프로젝트 내 best run을 찾아 에폭을 늘려 재학습
if 'wandb' in globals() and wandb is not None and os.environ.get("WANDB_MODE","").lower() != "offline":
    api = wandb.Api()
    # entity/project 자동 인식 (로그인 컨텍스트 기반)
    # 필요 시 'your_entity'로 교체: f"your_entity/{project_name}"
    runs = api.runs(f"{wandb.setup().entity}/{project_name}")
    best, best_val = None, float("inf")
    for r in runs:
        v = r.summary.get("best_val_loss", r.summary.get("val_loss", None))
        if v is not None and v < best_val:
            best_val = v; best = r

    if best is not None:
        print("Best run:", best.id, best.name, "best_val_loss:", best_val)
        best_cfg = dict(best.config)
        best_cfg["epochs"] = max(int(best_cfg.get("epochs", 50)), 100)  # 재학습 시 에폭 확장
        score, path, cfg = train_one_run(best_cfg)
        print("Re-run finished. best_score:", score, "path:", path)
    else:
        print("프로젝트에서 best run을 찾지 못했습니다.")
else:
    print("온라인 모드가 아니므로 Best 재실행을 건너뜁니다.")
